# Square-QDM evidence for Sec. 7

The main fixed-width logic now mirrors the spin-1 notebook: exact compact cage
and bounded operators, the **undeformed cage-excised microcanonical test** (T1),
finite-size matching to the resolved $\beta=0$ trace and transfer continuation
(T2), and only then nontrivial preserving deformations (T3).  The collective
quotient remains a secondary locality counterexample.


## Imports and run controls

The default run includes $4\times4$ and $8\times4$ microcanonical points.  Set `RUN_8X4_MICROCANONICAL=False` for a quick smoke test.  All draft figures are exported as PDF and SVG.

In [ ]:
from dataclasses import replace
from itertools import product
from pathlib import Path
import sys
import time

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.linalg as scipy_linalg
import scipy.sparse as scipy_sparse
from IPython.display import display

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "qlinks").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the qlinks repository root.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from helpers import (
    PRX_FOUR_PANEL_FIGSIZE,
    PRX_SINGLE_PANEL_FIGSIZE,
    PRX_TWO_PANEL_FIGSIZE,
    PRX_WIDE_FIGSIZE,
    add_panel_label,
    degeneracy_resolved_concentration,
    orthonormalize_columns,
    projector_deleted_basis,
    projector_deleted_concentration,
    projector_deleted_observable_moments,
    projector_resolved_energy_basis,
    save_prx_figure,
    set_revtex_matplotlib_style,
    use_integer_ticks,
    write_figure_manifest,
)

from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.builders import SparseHamiltonianBuilder
from qlinks.caging import (
    CageClassificationConfig,
    CageSearchConfig,
    CageSearcher,
    LocalQDMCageSearchConfig,
    LocalWitnessTemplate,
    Quasi1DSequencePoint,
    RobustQDMLocalCageSearchConfig,
    SquareQDMPeriodicProductUnitCell,
    SquareQDMStripTransferMatrix,
    SquareQDMWitnessPlacement,
    adjacent_gap_ratio_report,
    audit_quasi_1d_sequence,
    beta_zero_matching_subspace,
    cage_compatibility_hierarchy_from_hamiltonians,
    cage_finite_size_scorecard,
    cage_jacobian_conditioning_from_hamiltonian,
    certify_local_witness_on_square_qdm_periodic_sequence,
    certify_square_qdm_periodic_product_sequence,
    classify_cage_state,
    commuting_cyclic_symmetry_sector_basis,
    diagnose_boundary_cancellation_matroid,
    diagnose_eigenpair,
    diagnose_local_channel_spectrum,
    directed_transition_witness_template,
    eigenstate_expectations,
    evaluate_square_qdm_classification_witnesses_on_strips,
    gaussian_spectral_filter,
    local_witnesses_from_classification_report,
    materialize_square_qdm_periodic_product_state,
    operator_coefficient_compatibility,
    partition_cage_hamiltonian,
    product_basis_diagonal_phase_factors,
    project_coefficients_to_beta_zero_match,
    project_operator_to_sector,
    project_state_to_sector,
    regional_cage_quotient,
    robust_qdm_local_cage_search,
    scan_square_qdm_beta_zero_energy_density,
    scan_square_qdm_collective_locality_extension,
    scan_square_qdm_periodic_product_cancellation_scaling,
    scan_windowed_operator_annihilators,
    select_microcanonical_window_by_count,
    select_microcanonical_window_by_width,
    spectral_observable_moments,
    subspace_complement_basis,
    thermal_activity_margin_from_samples,
    thermodynamic_energy_window_plan,
)
from qlinks.models import (
    SquareQDMModel,
    qdm_peierls_couplings_from_link_phases,
    qdm_plaquette_link_gauge_matrix,
)
from qlinks.operators import PlaquettePatternOperator

TOL = 1.0e-10
RANK_TOL = 1.0e-9
RANDOM_SEED = 73291
USE_TEX = False
SAVE_FIGURES = True
SAVE_PDF = True
RUN_PROFILE = "smoke"  # "smoke", "known", or "production"
MICROCANONICAL_REPEAT_COUNTS_BY_PROFILE = {
    "smoke": (1,),
    "known": (1, 2),
    # Dense microcanonical ED for repeat=3 is memory-heavy even on a
    # large shared host.  Keep unattended production at repeats 1,2; use
    # the batch job's --ed-repeats 1,2,3 only when the 512 GB server is
    # effectively exclusive.
    "production": (1, 2),
}
PRODUCT_SCALING_REPEAT_COUNTS_BY_PROFILE = {
    "smoke": (1,),
    "known": (1, 2, 3),
    # These product/transfer-style diagnostics do not require dense ED, so
    # they can safely probe larger fixed-width sequences by default.
    "production": (1, 2, 3, 4),
}
PRODUCT_SCALING_MAX_SUPPORT_SIZE_BY_PROFILE = {
    "smoke": 128,
    "known": 128,
    "production": 256,
}
if RUN_PROFILE not in MICROCANONICAL_REPEAT_COUNTS_BY_PROFILE:
    raise ValueError(f"unknown RUN_PROFILE={RUN_PROFILE!r}")
MICROCANONICAL_REPEAT_COUNTS = MICROCANONICAL_REPEAT_COUNTS_BY_PROFILE[RUN_PROFILE]
PRODUCT_SCALING_REPEAT_COUNTS = PRODUCT_SCALING_REPEAT_COUNTS_BY_PROFILE[RUN_PROFILE]
PRODUCT_SCALING_MAX_SUPPORT_SIZE = PRODUCT_SCALING_MAX_SUPPORT_SIZE_BY_PROFILE[RUN_PROFILE]
RUN_8X4_MICROCANONICAL = max(MICROCANONICAL_REPEAT_COUNTS) >= 2
RUN_BACKGROUND_CONCENTRATION = True
RUN_NONUNIFORM_POTENTIAL_PATH = True
RUN_NON_GAUGE_KINETIC_PATH = True
RUN_COLLECTIVE_CLUSTER_SCAN = True
RUN_REVISED_Y_VALIDATION = True
STRICT_CLAIMS = False
RUN_BETA0_OVERLAP = True
EXCEPTIONAL_PROJECTOR_MODE = "identified-compact"  # or "target-only"
DEFORMATION_REFERENCE = "nonuniform-potential"

PEIERLS_REFERENCE_PHASE = 0.35
PEIERLS_PATH = np.linspace(0.15, 0.55, 5)
NONUNIFORM_POTENTIAL_REFERENCE = 0.20
MICROCANONICAL_PREFACTORS = (0.50, 0.75, 1.00)
PRIMARY_WINDOW_PREFACTOR = 0.75
SMOOTH_SIGMA_PREFACTOR = 0.75

DATA_DIR = REPO_ROOT / "experimental" / "data" / "square_qdm_draft_evidence"
FIGURE_DIR = DATA_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

set_revtex_matplotlib_style(base_font_size=9.0, prefer_tex=USE_TEX)

FIGURE_FORMATS = ("pdf", "svg")

def save_figure(fig, stem, *, aliases=(), close=False):
    save_prx_figure(fig, stem, directory=FIGURE_DIR, formats=FIGURE_FORMATS)
    for alias in aliases:
        if alias != stem:
            save_prx_figure(fig, alias, directory=FIGURE_DIR, formats=FIGURE_FORMATS)
    if close:
        plt.close(fig)


def embedded_state(record, hilbert_size):
    state = np.zeros(hilbert_size, dtype=np.complex128)
    state[np.asarray(record.cage_state.support, dtype=np.int64)] = record.cage_state.local_state
    return state


def square_qdm_basis_translation_permutation(model, basis_configs, *, dx=0, dy=0):
    """Return the basis permutation for a physical lattice translation."""
    link_lookup = {}
    for link in model.lattice.links:
        x, y = model.lattice.sites[int(link.source)].cell
        link_lookup[(int(x), int(y), str(link.kind))] = int(link.id)
    transformed = np.zeros_like(basis_configs)
    lx, ly = int(model.lattice.lx), int(model.lattice.ly)
    for link in model.lattice.links:
        x, y = model.lattice.sites[int(link.source)].cell
        target = link_lookup[((int(x) + dx) % lx, (int(y) + dy) % ly, str(link.kind))]
        transformed[:, target] = basis_configs[:, int(link.id)]
    lookup = {
        tuple(int(value) for value in config): index
        for index, config in enumerate(basis_configs)
    }
    return np.asarray(
        [lookup[tuple(int(value) for value in config)] for config in transformed],
        dtype=np.int64,
    )


def fixed_width_cage_momentum(repeats):
    """Momentum branch followed by the repeated compact product cage."""
    return 0, (2 * int(repeats)) % 4


print({
    "repository": str(REPO_ROOT),
    "data_directory": str(DATA_DIR),
    "run_profile": RUN_PROFILE,
    "microcanonical_repeat_counts": MICROCANONICAL_REPEAT_COUNTS,
    "product_scaling_repeat_counts": PRODUCT_SCALING_REPEAT_COUNTS,
    "reference_peierls_phase": PEIERLS_REFERENCE_PHASE,
})

## Evidence map and run products

The primary fixed-width family uses the witness order $A_R,Z_R,Y_R$.  The
revised shifted-shell $Y_R$ remains mandatory: if its resolved-cage darkness
fails, every main-text $Y$ column and curve is suppressed, while the older
equal-flippability difference is retained only as a secondary local-singlet
diagnostic.

- **T1 / C3a / C7:** undeformed, winding/momentum-resolved, cage-excised ETH
  scatter and fixed-width sequence.
- **T2 / C3b:** cage-excised--$\beta=0$ overlap on common sizes, followed by
  transfer counting.
- **T3 / C4--C6:** Peierls, non-gauge kinetic, and nonuniform-potential paths,
  visually and logically separate from the undeformed proof.


## 1. $4\times4$ Type-I cage census

We record the shell dimensions, boundary-matrix rank and nullity, support size, and full-Hamiltonian residual for the two Type-I manifolds used in the draft.

In [ ]:
square_model = SquareQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    winding_convention="electric",
    coup_kin=1.0,
    coup_pot=1.0,
)

t0 = time.perf_counter()
square_build = square_model.build(
    basis_solver="dfs",
    builder="sparse",
    backend="scipy",
    sort_basis=True,
)
build_seconds = time.perf_counter() - t0

t0 = time.perf_counter()
square_search = CageSearcher.from_model_build_result(
    square_build,
    config=CageSearchConfig(
        search_type="type1",
        tolerance=TOL,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=64,
        ipr_candidate_count=32,
        ipr_random_seed=1234,
    ),
).run()
search_seconds = time.perf_counter() - t0

records_04 = tuple(square_search[(0, 4)])
record_06 = square_search[(0, 6), 0]
states_04 = np.column_stack(
    [embedded_state(record, square_search.hilbert_size) for record in records_04]
)

{
    "winding_sector": (0, 0),
    "hilbert_dimension": square_search.hilbert_size,
    "counts_by_signature": square_search.counts_by_signature,
    "build_seconds": build_seconds,
    "search_seconds": search_seconds,
}

In [ ]:
scorecard_rows = []
for signature, records in (((0, 4), records_04), ((0, 6), (record_06,))):
    for record_index, record in enumerate(records):
        full_state = embedded_state(record, square_search.hilbert_size)
        scorecard = cage_finite_size_scorecard(
            square_build.hamiltonian,
            record.candidate.vertices,
            full_state,
            kinetic=square_build.kinetic,
            actual_support=record.cage_state.support,
            amplitude_tolerance=TOL,
            rank_tolerance=TOL,
            metadata={
                "signature": str(signature),
                "record": record_index,
                "basis_strategy": "IPR postselection",
                "winding_x": 0,
                "winding_y": 0,
            },
        )
        scorecard_rows.append(scorecard.to_summary_dict())

scorecard_table = pd.DataFrame(scorecard_rows)
scorecard_table.to_csv(DATA_DIR / "qdm_4x4_type1_scorecard.csv", index=False)
display(scorecard_table[[
    "signature", "record", "candidate_shell_size", "boundary_rows",
    "boundary_columns", "boundary_rank", "boundary_nullity",
    "boundary_singular_gap", "actual_support_size",
    "internal_residual", "boundary_residual", "relative_eigenpair_residual",
]])

The $(0,4)$ shell is a $84\times48$ boundary problem with nullity nine.  Eight IPR representatives have support four, while the ninth has support 48.  The $(0,6)$ shell is a $100\times32$ boundary problem with nullity one.

## 2. Compact and collective parts of the $(0,4)$ manifold

The eight support-four cages span the compact local sector.  Removing that span from the complete nine-dimensional $(0,4)$ cage manifold leaves one collective direction.

In [ ]:
regional_supports = tuple(record.cage_state.support for record in records_04[:8])
quotient_report = regional_cage_quotient(
    square_build.kinetic,
    regional_supports,
    states_04,
    tolerance=TOL,
)
quotient_overlaps = np.abs(states_04.conj().T @ quotient_report.quotient_basis) ** 2

full_support = tuple(
    sorted(set().union(*(set(record.cage_state.support) for record in records_04)))
)
full_blocks = partition_cage_hamiltonian(square_build.kinetic, full_support)
full_column = {basis_index: column for column, basis_index in enumerate(full_support)}
regional_columns = tuple(
    tuple(full_column[basis_index] for basis_index in support)
    for support in regional_supports
)
matroid_report = diagnose_boundary_cancellation_matroid(
    full_blocks.boundary,
    regional_columns,
    tolerance=TOL,
)

quotient_table = pd.DataFrame([{
    **quotient_report.to_summary_dict(),
    "collective_record_overlap": float(quotient_overlaps[8, 0]),
    "regional_circuit_count": matroid_report.regional_circuit_count,
    "weighted_relative_dependency_dimension": matroid_report.relative_dependency_dimension,
}])
quotient_table.to_csv(DATA_DIR / "qdm_4x4_compact_collective_quotient.csv", index=False)
display(quotient_table.T)

In [ ]:
fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
ax.bar(["complete cage\nmanifold", "compact span", "collective\nquotient"], [9, 8, 1])
ax.set_ylabel("Dimension")
ax.set_ylim(0, 10)
save_figure(fig, "qdm_4x4_9_equals_8_plus_1")
plt.show()

## Exact compact fixed-width family and transported local motif

In [ ]:
local_search_config = LocalQDMCageSearchConfig(
    halo_layers=0,
    boundary_mode="relaxed",
    prune_inactive_local_basis_states=True,
    tolerance=TOL,
    degenerate_basis_strategy="ipr",
    ipr_random_seed=1234,
)
robust_config = RobustQDMLocalCageSearchConfig(
    local_config=local_search_config,
    region_strategies=("stripe",),
    stripe_widths=(1,),
    stripe_directions=(0, 1),
    max_regions_per_strategy=None,
    block_signatures=((0, 2),),
    max_records_per_region=2,
    min_blocks=2,
    max_blocks=None,
    max_product_support_size=2048,
    max_paddings_per_stage=100,
    max_paddings_per_packing=10,
    include_sectors=True,
    padding_stages=("static",),
    tolerance=1.0e-9,
    store_full_states=False,
)

stripe_certified, stripe_context = robust_qdm_local_cage_search(
    square_model,
    config=robust_config,
    return_context=True,
)

repeatable_candidates = []
for report_index, report in enumerate(stripe_certified.reports):
    try:
        candidate = SquareQDMPeriodicProductUnitCell.from_padding(
            square_model,
            stripe_context.blocks,
            report.padding,
            repeat_axis="x",
        )
        certificate = certify_square_qdm_periodic_product_sequence(candidate)
    except ValueError:
        continue
    if certificate.is_certified:
        repeatable_candidates.append((report_index, candidate, certificate))

if not repeatable_candidates:
    raise RuntimeError("No x-repeatable square-QDM product unit cell was found.")

repeatable_report_index, product_unit_cell, product_sequence = repeatable_candidates[0]
{
    "is_certified": product_sequence.is_certified,
    "repeat_axis": product_unit_cell.repeat_axis,
    "energy_density": product_sequence.energy_density,
    "support_size_per_unit_cell": product_unit_cell.support_size_per_unit_cell,
    "unit_cell_winding_sector": product_sequence.unit_cell_winding_sector,
    "verification_repeats": product_sequence.verification_repeats,
}

peierls_product_unit_cell = product_unit_cell.with_couplings(
    coup_kin=np.exp(1.0j * PEIERLS_REFERENCE_PHASE),
    coup_pot=1.0,
)
peierls_product_sequence = certify_square_qdm_periodic_product_sequence(
    peierls_product_unit_cell,
    tolerance=1.0e-9,
)
if not peierls_product_sequence.is_certified:
    raise RuntimeError("The reference Peierls sequence failed exact certification.")
print({
    "peierls_phase": PEIERLS_REFERENCE_PHASE,
    "peierls_sequence_certified": peierls_product_sequence.is_certified,
    "peierls_energy_density": peierls_product_sequence.energy_density,
})


In [ ]:
product_scaling = scan_square_qdm_periodic_product_cancellation_scaling(
    product_unit_cell,
    repeat_counts=PRODUCT_SCALING_REPEAT_COUNTS,
    max_support_size=PRODUCT_SCALING_MAX_SUPPORT_SIZE,
    tolerance=1.0e-9,
)
product_scaling_table = pd.DataFrame([
    point.to_summary_dict() for point in product_scaling.points
])
product_scaling_table.to_csv(DATA_DIR / "qdm_4N_by_4_exact_sequence.csv", index=False)
display(product_scaling_table[[
    "repeats", "system_size", "support_size", "shell_size",
    "boundary_nullity", "interference_gap", "product_state_boundary_residual",
    "kinetic_constraint_rank", "kinetic_compatible_dimension",
    "kinetic_compatible_fraction", "potential_constraint_rank",
]])

In [ ]:
fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
ax.plot(product_scaling_table["repeats"], product_scaling_table["interference_gap"], marker="o")
ax.set_xlabel("Number of repeated unit cells $N$")
ax.set_ylabel(r"Boundary singular gap $\Delta_B$")
ax.set_xticks(product_scaling_table["repeats"])
save_figure(fig, "qdm_strip_interference_gap")
plt.show()

fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
ax.plot(product_scaling_table["repeats"], product_scaling_table["kinetic_constraint_rank"], marker="o", label="compatibility rank")
ax.plot(product_scaling_table["repeats"], [16*n for n in product_scaling_table["repeats"]], marker="o", label="local kinetic parameters")
ax.set_xlabel("Number of repeated unit cells $N$")
ax.set_ylabel("Dimension")
ax.set_xticks(product_scaling_table["repeats"])
ax.legend(frameon=False)
save_figure(fig, "qdm_strip_compatibility_scaling")
plt.show()

The repeated cage remains exact on the tested sizes.  Its local cancellation rule is size independent, although the number of independent local coupling constraints grows with the strip length.  This distinction is kept separate from the ETH test below.

## C1--C2. Bounded $A_R$, $Z_R$, and revised shifted-potential $Y_R$

In [ ]:
stripe_record = stripe_certified.records[repeatable_report_index]
stripe_classification = classify_cage_state(
    stripe_record.cage_state,
    kinetic_matrix=stripe_certified.kinetic_matrix,
    basis_configs=stripe_certified.basis.states,
    hilbert_size=stripe_certified.hilbert_size,
    config=CageClassificationConfig(sector_policy="infer_support_component"),
)
stripe_witnesses = local_witnesses_from_classification_report(stripe_classification)

# Z_R: select the first certified Hermitian local interference witness and
# normalize it at its final operator support.
strip_lengths = (4, 8, 12, 16, 24, 32, 48, 64, 96, 128)
strip_witness_report = evaluate_square_qdm_classification_witnesses_on_strips(
    stripe_classification,
    model=square_model,
    lengths=strip_lengths,
    winding_sector=(0, 0),
    normalization="operator_norm",
    winding_projection="fourier",
)
selected_strip_witness = strip_witness_report.records[0]
z_reference_witness = selected_strip_witness.witness
z_placement = selected_strip_witness.placement


def directed_witness_from_hermitian_star(z_witness):
    operator = np.asarray(z_witness.template.local_operator, dtype=np.complex128)
    adjacency = np.abs(operator) > TOL
    degrees = np.sum(adjacency, axis=1)
    target_index = int(np.argmax(degrees))
    source_indices = np.flatnonzero(adjacency[target_index])
    if source_indices.size < 2:
        raise RuntimeError("Expected a two-parent local interference witness.")
    template = directed_transition_witness_template(
        target_pattern=z_witness.template.local_patterns[target_index],
        source_patterns=[z_witness.template.local_patterns[index] for index in source_indices],
        amplitudes=[operator[target_index, index] for index in source_indices],
        metadata={"name": "A_R", "source": "QDM compact-cage boundary row"},
        normalization="operator_norm",
    )
    return template.instantiate(z_witness.variable_indices)


def plaquette_local_positions(model, global_variable_indices, anchor):
    plaquette_id = next(
        int(pid)
        for pid in model.plaquette_ids()
        if tuple(model.lattice.plaquette_anchor_cell(int(pid))) == tuple(anchor)
    )
    position = {int(variable): index for index, variable in enumerate(global_variable_indices)}
    plaquette_variables = [
        int(model.layout.link_variable_index(int(link_id)))
        for link_id in model.lattice.plaquette_links(plaquette_id)
    ]
    return tuple(position[variable] for variable in plaquette_variables)


def flippability_on_pattern(pattern, positions):
    local = tuple(pattern[index] for index in positions)
    return 1.0 if local in ((1, 0, 1, 0), (0, 1, 0, 1)) else 0.0


def shifted_potential_witness(z_witness, *, mode):
    """Construct the revised shell generator or the older equality diagnostic."""
    p1_anchor, p2_anchor = (0, 0), (0, 2)
    p1 = plaquette_local_positions(square_model, z_witness.variable_indices, p1_anchor)
    p2 = plaquette_local_positions(square_model, z_witness.variable_indices, p2_anchor)
    local_patterns = tuple(product((0, 1), repeat=z_witness.template.n_variables))
    if mode == "shell":
        diagonal = np.asarray(
            [0.5 * (flippability_on_pattern(pattern, p1) + flippability_on_pattern(pattern, p2) - 2.0) for pattern in local_patterns],
            dtype=np.complex128,
        )
        definition = "0.5*(F_(0,0)+F_(0,2)-2I)"
        name = "Y_R_shell"
    elif mode == "equal":
        diagonal = np.asarray(
            [flippability_on_pattern(pattern, p1) - flippability_on_pattern(pattern, p2) for pattern in local_patterns],
            dtype=np.complex128,
        )
        definition = "F_(0,0)-F_(0,2)"
        name = "Y_R_equal_flippability"
    else:
        raise ValueError(mode)
    template = LocalWitnessTemplate(
        pattern_key=(),
        local_patterns=local_patterns,
        local_operator=np.diag(diagonal),
        metadata={
            "name": name,
            "definition": definition,
            "plaquette_anchors": (p1_anchor, p2_anchor),
            "potential_shell_value": 2.0 if mode == "shell" else None,
        },
    )
    return template.instantiate(z_witness.variable_indices)


a_reference_witness = directed_witness_from_hermitian_star(z_reference_witness)
a_placement = SquareQDMWitnessPlacement.from_local_witness(square_model, a_reference_witness)
y_equal_reference_witness = shifted_potential_witness(z_reference_witness, mode="equal")
y_shell_reference_witness = shifted_potential_witness(z_reference_witness, mode="shell")
y_equal_placement = SquareQDMWitnessPlacement.from_local_witness(square_model, y_equal_reference_witness)
y_placement = SquareQDMWitnessPlacement.from_local_witness(square_model, y_shell_reference_witness)

# Seven-step revised-Y validation: identify the plaquettes, certify the sharp
# support value configuration by configuration, compare old/new residuals,
# transport the same rule through the repeated compact sequence, and only then
# enable Y-dependent thermal outputs.
y_support_rows = []
y_size_rows = []
Y_SHELL_VALID = True
for repeats in PRODUCT_SCALING_REPEAT_COUNTS:
    raw_instance = product_unit_cell.instantiate(int(repeats))
    finite_model = replace(raw_instance.model, winding_x=0, winding_y=0)
    instance = replace(raw_instance, model=finite_model)
    build = finite_model.build(basis_solver="dfs", builder="sparse", backend="scipy", sort_basis=True)
    configs = basis_configs_from_build_result(build)
    cage = materialize_square_qdm_periodic_product_state(instance, configs)
    support = np.flatnonzero(np.abs(cage) > TOL)
    y_shell = y_placement.instantiate_on_model(finite_model)
    y_equal = y_equal_placement.instantiate_on_model(finite_model)
    y_shell_op = y_shell.embed(configs)
    y_equal_op = y_equal.embed(configs)
    shell_residual = float(np.linalg.norm(y_shell_op @ cage))
    equal_residual = float(np.linalg.norm(y_equal_op @ cage))
    shell_activity = float(np.vdot(cage, (y_shell_op.conj().T @ y_shell_op) @ cage).real)
    # A localized compact cage is sharp on the selected plaquettes.  The
    # translation-projected momentum eigenstate used by a translation-invariant
    # thermal reference need not remain locally sharp; test it separately.
    tx = square_qdm_basis_translation_permutation(finite_model, configs, dx=1)
    ty = square_qdm_basis_translation_permutation(finite_model, configs, dy=1)
    kx, ky = fixed_width_cage_momentum(repeats)
    sector = commuting_cyclic_symmetry_sector_basis(
        (tx, ty), orders=(finite_model.lx, finite_model.ly), momentum_indices=(kx, ky)
    )
    projected_cage = project_state_to_sector(cage, sector)
    projected_cage /= np.linalg.norm(projected_cage)
    projected_y = project_operator_to_sector(y_shell_op, sector)
    projected_shell_residual = float(np.linalg.norm(projected_y @ projected_cage))
    projected_shell_activity = float(np.vdot(projected_cage, (projected_y.conj().T @ projected_y) @ projected_cage).real)
    localized_valid = shell_residual <= 1.0e-9 and abs(shell_activity) <= 1.0e-9
    resolved_valid = projected_shell_residual <= 1.0e-9 and abs(projected_shell_activity) <= 1.0e-9
    Y_SHELL_VALID &= localized_valid and resolved_valid
    y_size_rows.append({
        "repeats": int(repeats), "Lx": int(finite_model.lx), "Ly": int(finite_model.ly),
        "support_size": int(support.size), "Y_equal_residual": equal_residual,
        "Y_shell_localized_residual": shell_residual, "QY_shell_localized_cage": shell_activity,
        "Y_shell_resolved_residual": projected_shell_residual,
        "QY_shell_resolved_cage": projected_shell_activity,
        "localized_shell_valid": localized_valid,
        "resolved_shell_valid": resolved_valid,
    })
    # Configuration-level sharp shell values for the transported plaquettes.
    for basis_index in support:
        config = configs[int(basis_index)]
        values = []
        for anchor in ((0, 0), (0, 2)):
            pid = next(int(pid) for pid in finite_model.plaquette_ids() if tuple(finite_model.lattice.plaquette_anchor_cell(int(pid))) == anchor)
            variables = [int(finite_model.layout.link_variable_index(int(link_id))) for link_id in finite_model.lattice.plaquette_links(pid)]
            pattern = tuple(int(config[index]) for index in variables)
            values.append(1 if pattern in ((1,0,1,0),(0,1,0,1)) else 0)
        y_support_rows.append({
            "repeats": int(repeats), "Lx": int(finite_model.lx), "basis_index": int(basis_index),
            "amplitude_abs": float(abs(cage[int(basis_index)])), "F_p1": values[0], "F_p2": values[1],
            "V_R": int(values[0] + values[1]), "shell_condition": bool(values == [1, 1]),
        })
        Y_SHELL_VALID &= values == [1, 1]

y_support_table = pd.DataFrame(y_support_rows)
y_size_validation_table = pd.DataFrame(y_size_rows)
y_support_table.to_csv(DATA_DIR / "qdm_revised_Y_support_values.csv", index=False)
y_size_validation_table.to_csv(DATA_DIR / "qdm_revised_Y_size_validation.csv", index=False)
Y_THERMAL_ENABLED = bool(Y_SHELL_VALID)
Y_VALIDATION_STATUS = "established" if Y_THERMAL_ENABLED else "localized_only_not_dark_after_symmetry_projection"
if RUN_REVISED_Y_VALIDATION and STRICT_CLAIMS and not Y_THERMAL_ENABLED:
    raise RuntimeError("The revised shifted-potential shell operator is not dark in the resolved cage used by T1/T2.")
if not Y_THERMAL_ENABLED:
    print(
        "Revised Y_R is sharp for the localized compact cage but not for the "
        "translation-projected cage. T1/T2 figures and tables will therefore "
        "report A_R,Z_R only; Y_R remains available on translation-breaking paths."
    )
MAIN_WITNESS_NAMES = ("A", "Z") + (("Y",) if Y_THERMAL_ENABLED else ())
display(y_size_validation_table)

# Verify all three channels before and after the controlled Peierls reference.
sequence_certificates = []
for label, witness in (("A", a_reference_witness), ("Z", z_reference_witness), ("Y_shell", y_shell_reference_witness), ("Y_equal_secondary", y_equal_reference_witness)):
    base_certificate = certify_local_witness_on_square_qdm_periodic_sequence(product_sequence, witness, normalization="operator_norm")
    phase_certificate = certify_local_witness_on_square_qdm_periodic_sequence(peierls_product_sequence, witness, normalization="operator_norm")
    sequence_certificates.append({
        "witness": label,
        "base_residual": base_certificate.annihilation_residual,
        "peierls_residual": phase_certificate.annihilation_residual,
        "Q_norm": base_certificate.witness.q_operator_norm,
        "main_text_route": label in {"A", "Z", "Y_shell"},
    })
three_witness_certificate_table = pd.DataFrame(sequence_certificates)
three_witness_certificate_table.to_csv(DATA_DIR / "qdm_three_witness_certificates.csv", index=False)
display(three_witness_certificate_table)

### Size-compatible real-space operator placement

The same microscopic operators are transported to every $(4N)\times4$ member by the placement objects below. The table records their support variables and plaquette anchors so that the fixed-width claim does not rely on refitting a new annihilator at each size.

In [ ]:
operator_transport_table = pd.DataFrame(
    [
        {
            "witness": "A",
            "definition": "directed two-parent boundary row",
            "window_width": int(a_placement.window_width),
            "circumference": int(a_placement.circumference),
            "reference_origin_x": a_placement.reference_origin_x,
            "link_coordinates": repr(a_placement.link_coordinates),
        },
        {
            "witness": "Z",
            "definition": "A+A^dagger on the same stripe cluster",
            "window_width": int(z_placement.window_width),
            "circumference": int(z_placement.circumference),
            "reference_origin_x": z_placement.reference_origin_x,
            "link_coordinates": repr(z_placement.link_coordinates),
        },
        {
            "witness": "Y",
            "definition": "0.5*[F_(0,0)+F_(0,2)-2I] on the same stripe cluster",
            "window_width": int(y_placement.window_width),
            "circumference": int(y_placement.circumference),
            "reference_origin_x": y_placement.reference_origin_x,
            "link_coordinates": repr(y_placement.link_coordinates),
        },
    ]
)
operator_transport_table.to_csv(DATA_DIR / "qdm_three_witness_operator_transport.csv", index=False)
display(operator_transport_table)

## T1. Undeformed cage-excised fixed-width microcanonical comparison

In [ ]:
def identified_qdm_exceptional_vectors(*, repeats, finite_model, configs, sector, target_energy, target_cage):
    candidates = [project_state_to_sector(target_cage, sector)]
    labels = ["selected_repeated_compact_cage"]
    if EXCEPTIONAL_PROJECTOR_MODE == "identified-compact":
        if int(repeats) == 1:
            # Complete 4x4 Type-I (0,4) manifold from the exhaustive shell solve.
            for record_index, record in enumerate(records_04):
                state = embedded_state(record, square_search.hilbert_size)
                projected = project_state_to_sector(state, sector)
                if np.linalg.norm(projected) > TOL:
                    candidates.append(projected)
                    labels.append(f"4x4_type1_04_{record_index}")
        else:
            # Every independently certified repeatable compact product unit cell
            # available from the local search is transported to this strip and
            # projected to the same momentum branch.  This is the complete
            # currently identified compact family, not a claim of exhaustive
            # classification of all possible QDM cages at larger width.
            for candidate_index, (_report_index, candidate, _certificate) in enumerate(repeatable_candidates):
                try:
                    raw = candidate.instantiate(int(repeats))
                    model = replace(raw.model, winding_x=0, winding_y=0)
                    instance = replace(raw, model=model)
                    if model.lattice.lx != finite_model.lattice.lx:
                        continue
                    state = materialize_square_qdm_periodic_product_state(instance, configs)
                except Exception:
                    continue
                projected = project_state_to_sector(state, sector)
                if np.linalg.norm(projected) > TOL:
                    candidates.append(projected)
                    labels.append(f"repeatable_compact_{candidate_index}")
    matrix = orthonormalize_columns(np.column_stack(candidates), tolerance=1.0e-9)
    h_full = finite_model.build(basis_solver="dfs", builder="bitmask", backend="scipy", sort_basis=True).hamiltonian
    h_sector_check = project_operator_to_sector(h_full, sector)
    residuals=[]
    accepted=[]
    for column in range(matrix.shape[1]):
        state=matrix[:,column]
        residual=float(np.linalg.norm(h_sector_check@state-target_energy*state))
        if residual <= 1.0e-8:
            accepted.append(state)
            residuals.append(residual)
    matrix=orthonormalize_columns(np.column_stack(accepted),tolerance=1.0e-9) if accepted else np.zeros((sector.sector_dimension,0),dtype=np.complex128)
    return matrix, {"projector_rank":int(matrix.shape[1]),"candidate_count":len(candidates),
                    "max_residual":max(residuals,default=0.0),"labels":";".join(labels),
                    "mode":EXCEPTIONAL_PROJECTOR_MODE}


def fixed_width_microcanonical_point(repeats):
    raw_instance = product_unit_cell.with_couplings(coup_kin=1.0, coup_pot=1.0).instantiate(int(repeats))
    finite_model = replace(raw_instance.model, winding_x=0, winding_y=0)
    instance = replace(raw_instance, model=finite_model)
    t0 = time.perf_counter()
    print(f"  building undeformed {finite_model.lx}x{finite_model.ly} zero-winding sector ...", flush=True)
    build = finite_model.build(basis_solver="dfs", builder="bitmask", backend="scipy", sort_basis=True)
    configs = basis_configs_from_build_result(build)
    cage = materialize_square_qdm_periodic_product_state(instance, configs)
    cage_report = diagnose_eigenpair(build.hamiltonian, cage)
    target_energy = float(cage_report.energy.real)

    tx = square_qdm_basis_translation_permutation(finite_model, configs, dx=1)
    ty = square_qdm_basis_translation_permutation(finite_model, configs, dy=1)
    kx, ky = fixed_width_cage_momentum(repeats)
    sector = commuting_cyclic_symmetry_sector_basis(
        (tx, ty), orders=(finite_model.lx, finite_model.ly), momentum_indices=(kx, ky),
        labels={"winding_x":0,"winding_y":0,"kx_index":kx,"ky_index":ky},
    )
    cage_sector = project_state_to_sector(cage, sector)
    projection_norm=float(np.linalg.norm(cage_sector))
    if projection_norm <= TOL: raise RuntimeError("predicted momentum branch has zero cage weight")
    cage_sector/=projection_norm
    h_sector=project_operator_to_sector(build.hamiltonian,sector)
    energies,vectors=scipy_linalg.eigh(h_sector,check_finite=False)

    z_witness=z_placement.instantiate_on_model(finite_model)
    a_witness=a_placement.instantiate_on_model(finite_model)
    y_witness=y_placement.instantiate_on_model(finite_model) if Y_THERMAL_ENABLED else None
    local={"A":a_witness.embed(configs),"Z":z_witness.embed(configs)}
    if y_witness is not None: local["Y"]=y_witness.embed(configs)
    q_sector={name:project_operator_to_sector(op.conj().T@op,sector) for name,op in local.items()}
    z_sector=project_operator_to_sector(local["Z"],sector)
    y_sector=project_operator_to_sector(local["Y"],sector) if y_witness is not None else None
    exceptional, projector_meta = identified_qdm_exceptional_vectors(
        repeats=repeats, finite_model=finite_model, configs=configs, sector=sector,
        target_energy=target_energy, target_cage=cage,
    )

    q_expect={name:eigenstate_expectations(op,vectors) for name,op in q_sector.items()}
    z_expect=eigenstate_expectations(z_sector,vectors)
    y_expect=eigenstate_expectations(y_sector,vectors) if y_sector is not None else np.full(energies.shape,np.nan)
    rows=[]; concentration_rows=[]; primary=None
    for prefactor in MICROCANONICAL_PREFACTORS:
        plan=thermodynamic_energy_window_plan(volume=finite_model.lattice.num_plaquettes,
            energy_density=target_energy/finite_model.lattice.num_plaquettes,
            width_prefactor=prefactor,local_energy_scale=1.0)
        window=select_microcanonical_window_by_width(energies,target_energy=target_energy,
            half_width=plan.half_width,degeneracy_tolerance=TOL)
        indices=np.asarray(window.indices,dtype=np.int64)
        split=projector_deleted_basis(vectors[:,indices],exceptional,tolerance=1.0e-9)
        moments={name:projector_deleted_observable_moments(vectors[:,indices],exceptional,op,
                    squared_operator=op,tolerance=1.0e-9) for name,op in q_sector.items()}
        z_mean=projector_deleted_observable_moments(vectors[:,indices],exceptional,z_sector,
                    squared_operator=q_sector["Z"],tolerance=1.0e-9)
        y_mean=(projector_deleted_observable_moments(vectors[:,indices],exceptional,y_sector,
                    squared_operator=q_sector["Y"],tolerance=1.0e-9) if y_sector is not None else None)
        row={"repeats":int(repeats),"Lx":int(finite_model.lx),"Ly":int(finite_model.ly),
             "volume":int(finite_model.lattice.num_plaquettes),"winding_x":0,"winding_y":0,
             "kx_index":int(kx),"ky_index":int(ky),"resolved_sector_dimension":int(sector.sector_dimension),
             "reference":"undeformed","cage_energy":target_energy,
             "cage_energy_density":target_energy/finite_model.lattice.num_plaquettes,
             "cage_residual":float(cage_report.residual_norm),"window_prefactor":float(prefactor),
             "window_half_width":float(window.half_width),
             "window_energy_density_half_width":float(window.half_width/finite_model.lattice.num_plaquettes),
             "window_state_count":int(window.n_states),"retained_state_count":int(split["retained_rank"]),
             "removed_projector_rank":int(split["exceptional_rank"]),"removed_fraction":float(split["removed_fraction"]),
             "exceptional_projector_mode":EXCEPTIONAL_PROJECTOR_MODE,
             "projector_candidate_count":int(projector_meta["candidate_count"]),
             "projector_residual":float(projector_meta["max_residual"]),
             "thermal_A_activity":float(moments["A"]["mean"]),
             "thermal_Z_activity":float(moments["Z"]["mean"]),
             "thermal_Y_activity":float(moments["Y"]["mean"]) if "Y" in moments else np.nan,
             "cage_Z_mean":float(np.vdot(cage_sector,z_sector@cage_sector).real),
             "cage_Y_mean":float(np.vdot(cage_sector,y_sector@cage_sector).real) if y_sector is not None else np.nan,
             "thermal_Z_mean":float(z_mean["mean"]),"thermal_Z_variance":float(z_mean["variance"]),
             "thermal_Y_mean":float(y_mean["mean"]) if y_mean else np.nan,
             "thermal_Y_variance":float(y_mean["variance"]) if y_mean else np.nan,
             "cage_QA":float(np.vdot(cage_sector,q_sector["A"]@cage_sector).real),
             "cage_QZ":float(np.vdot(cage_sector,q_sector["Z"]@cage_sector).real),
             "cage_QY":float(np.vdot(cage_sector,q_sector["Y"]@cage_sector).real) if "Y" in q_sector else np.nan,
             "Y_validation_status":Y_VALIDATION_STATUS,"runtime_seconds":time.perf_counter()-t0}
        rows.append(row)
        if abs(prefactor-PRIMARY_WINDOW_PREFACTOR)<=TOL: primary=row; primary_window=window
        if RUN_BACKGROUND_CONCENTRATION:
            diagnostics={"Z_R":z_sector}
            if y_sector is not None: diagnostics["Y_R"]=y_sector
            for variable in tuple(int(v) for v in z_witness.variable_indices):
                n_full=scipy_sparse.diags(np.asarray(configs[:,variable],dtype=np.float64),format="csr")
                diagnostics[f"n_link_{variable}"]=project_operator_to_sector(n_full,sector)
            for name,operator in diagnostics.items():
                norm=float(np.max(np.abs(np.linalg.eigvalsh(0.5*(operator+operator.conj().T)))))
                if norm<=TOL: continue
                diagnostic=projector_deleted_concentration(vectors[:,indices],exceptional,operator/norm,tolerance=1.0e-9)
                concentration_rows.append({"repeats":int(repeats),"Lx":int(finite_model.lx),"Ly":int(finite_model.ly),
                                           "window_prefactor":float(prefactor),"operator":name,**diagnostic})
    if primary is None: raise RuntimeError("primary window missing")

    resolved=projector_resolved_energy_basis(energies,vectors,exceptional,energy_tolerance=1.0e-9,vector_tolerance=1.0e-9)
    values={name:np.real(np.einsum("ij,ij->j",resolved["basis"].conj(),op@resolved["basis"])) for name,op in q_sector.items()}
    scatter=pd.DataFrame({"repeats":int(repeats),"Lx":int(finite_model.lx),"Ly":int(finite_model.ly),
                          "energy":resolved["energies"],"energy_density":resolved["energies"]/finite_model.lattice.num_plaquettes,
                          "is_exceptional":resolved["is_exceptional"],
                          "is_primary_window":np.abs(resolved["energies"]-target_energy)<=primary_window.half_width+TOL,
                          "Q_A":values["A"],"Q_Z":values["Z"],"Q_Y":values.get("Y",np.full(resolved["energies"].shape,np.nan))})
    cache={"energies":energies,"vectors":vectors,"exceptional":exceptional,"q_sector":q_sector,
           "target_energy":target_energy,"sector":sector,"model":finite_model}
    return rows,primary,scatter,concentration_rows,cache


fixed_width_rows=[]; fixed_width_primary=[]; fixed_width_scatter=[]; fixed_width_concentration=[]; qdm_primary_cache={}
for repeats in MICROCANONICAL_REPEAT_COUNTS:
    rows,primary,scatter,concentration,cache=fixed_width_microcanonical_point(repeats)
    fixed_width_rows.extend(rows); fixed_width_primary.append(primary); fixed_width_scatter.append(scatter)
    fixed_width_concentration.extend(concentration); qdm_primary_cache[int(repeats)]=cache
fixed_width_window_table=pd.DataFrame(fixed_width_rows)
fixed_width_primary_table=pd.DataFrame(fixed_width_primary)
fixed_width_scatter_table=pd.concat(fixed_width_scatter,ignore_index=True)
fixed_width_concentration_table=pd.DataFrame(fixed_width_concentration)
fixed_width_window_table.to_csv(DATA_DIR/"qdm_cage_excised_fixed_width.csv",index=False)
fixed_width_primary_table.to_csv(DATA_DIR/"qdm_fixed_width_microcanonical_primary.csv",index=False)
fixed_width_scatter_table.to_csv(DATA_DIR/"qdm_cage_excised_eth_scatter.csv",index=False)
fixed_width_scatter_table.to_csv(DATA_DIR/"qdm_fixed_width_eth_scatter.csv",index=False)
fixed_width_concentration_table.to_csv(DATA_DIR/"qdm_cage_excised_concentration.csv",index=False)
fixed_width_concentration_table.to_csv(DATA_DIR/"qdm_background_concentration.csv",index=False)
display(fixed_width_primary_table)


In [ ]:
# QDM analogue of the spin-1 undeformed ETH scatter and finite-size panel.
central=fixed_width_primary_table.sort_values("Lx")
largest_Lx=int(central["Lx"].max())
scatter_largest=fixed_width_scatter_table[fixed_width_scatter_table["Lx"]==largest_Lx]
primary_largest=central[central["Lx"]==largest_Lx].iloc[0]
fig=plt.figure(figsize=PRX_TWO_PANEL_FIGSIZE)
gs=fig.add_gridspec(1,2,left=0.09,right=0.985,bottom=0.18,top=0.88,wspace=0.30)
ax=fig.add_subplot(gs[0,0])
center=float(primary_largest["cage_energy_density"]); half=float(primary_largest["window_energy_density_half_width"])
ax.axvspan(center-half,center+half,color="0.5",alpha=0.10,zorder=0); ax.axvline(center,color="0.45",ls="--",lw=0.8)
specs=[("Q_A",r"$Q_R^A$","o"),("Q_Z",r"$Q_R^Z$","s")]+([("Q_Y",r"$Q_R^Y$","^")] if Y_THERMAL_ENABLED else [])
for column,label,marker in specs:
    retained=scatter_largest[~scatter_largest["is_exceptional"]]; removed=scatter_largest[scatter_largest["is_exceptional"]]
    ax.scatter(retained["energy_density"],retained[column],s=9,alpha=0.50,marker=marker,label=label)
    ax.scatter(removed["energy_density"],removed[column],s=20,facecolors="none",edgecolors="0.25",marker=marker)
ax.scatter([center],[0.0],marker="*",s=70,edgecolors="black",linewidths=0.4,label="selected cage",zorder=5)
ax.set_xlabel(r"Energy density $e=E/(L_xL_y)$"); ax.set_ylabel("Local witness activity"); ax.grid(alpha=0.22); add_panel_label(ax,"(a)")
ax2=fig.add_subplot(gs[0,1])
plot_specs=[("thermal_A_activity",r"$Q_R^A$","o"),("thermal_Z_activity",r"$Q_R^Z$","s")]+([("thermal_Y_activity",r"$Q_R^Y$","^")] if Y_THERMAL_ENABLED else [])
for column,label,marker in plot_specs:
    group=fixed_width_window_table.groupby("Lx")[column]
    vals=central[column].to_numpy(); lows=group.min().reindex(central["Lx"]).to_numpy(); highs=group.max().reindex(central["Lx"]).to_numpy()
    ax2.errorbar(central["Lx"],vals,yerr=np.vstack([vals-lows,highs-vals]),marker=marker,capsize=2.5,label=label)
ax2.set_xlabel(r"Strip length $L_x$"); ax2.set_ylabel(r"$\tau_Q^{\rm mc,th}$"); ax2.grid(alpha=0.22); add_panel_label(ax2,"(b)")
handles,labels=ax.get_legend_handles_labels(); fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(0.5,0.99),ncol=4)
save_figure(fig,"qdm_undeformed_eth_scatter_and_sequence",aliases=("qdm_fixed_width_eth_scatter_largest",))
plt.show()


### T1a. Hermitian mean--fluctuation resolution

In [ ]:
hermitian_fixed_width_table = fixed_width_primary_table[
    [
        "Lx", "Ly", "cage_Z_mean", "cage_Y_mean",
        "thermal_Z_mean", "thermal_Z_variance", "thermal_Z_activity",
        "thermal_Y_mean", "thermal_Y_variance", "thermal_Y_activity",
    ]
].copy()
hermitian_fixed_width_table.to_csv(DATA_DIR / "qdm_fixed_width_hermitian_mean_variance.csv", index=False)
display(hermitian_fixed_width_table)

fig = plt.figure(figsize=PRX_TWO_PANEL_FIGSIZE)
grid = fig.add_gridspec(1, 2, wspace=0.30)
ax_z = fig.add_subplot(grid[0, 0])
ax_y = fig.add_subplot(grid[0, 1])

ax_z.plot(hermitian_fixed_width_table["Lx"], hermitian_fixed_width_table["thermal_Z_mean"], marker="o", label=r"$\langle Z_R\rangle_{\rm mc}$")
ax_z.plot(hermitian_fixed_width_table["Lx"], hermitian_fixed_width_table["thermal_Z_variance"], marker="s", label=r"${\rm Var}_{\rm mc}(Z_R)$")
ax_z.axhline(0.0, linestyle="--", linewidth=0.8, label="cage")
ax_z.set_xlabel(r"Strip length $L_x$")
ax_z.set_ylabel("Hermitian mean / variance")
ax_z.legend(loc="upper right")
ax_z.grid(alpha=0.25)
add_panel_label(ax_z, "(a)")

if Y_THERMAL_ENABLED:
    ax_y.plot(hermitian_fixed_width_table["Lx"], hermitian_fixed_width_table["thermal_Y_mean"], marker="o", label=r"$\langle Y_R\rangle_{\rm mc}$")
    ax_y.plot(hermitian_fixed_width_table["Lx"], hermitian_fixed_width_table["thermal_Y_variance"], marker="s", label=r"${\rm Var}_{\rm mc}(Y_R)$")
    ax_y.axhline(0.0, linestyle="--", linewidth=0.8, label="cage")
    ax_y.legend(loc="upper right")
else:
    ax_y.text(0.5, 0.5, r"$Y_R^{\rm shell}$ not dark after symmetry projection", ha="center", va="center", transform=ax_y.transAxes)
ax_y.set_xlabel(r"Strip length $L_x$")
ax_y.set_ylabel("Hermitian mean / variance")
ax_y.grid(alpha=0.25)
add_panel_label(ax_y, "(b)")

fig.tight_layout()
save_figure(fig, "qdm_fixed_width_hermitian_mean_variance")
plt.show()


The fixed-width calculation always reports the controlled kinetic routes $A_R^\dagger A_R$ and $Z_R^2$. The revised shell operator $Y_R$ is included only when it remains dark in the same symmetry-resolved cage used by the thermal comparison. The validation table therefore determines whether the QDM T1/T2 result is a two- or three-witness statement; additional $L_x$ values are still required for a limiting claim.

### T1b. Background concentration in the resolved fixed-width sector

In [ ]:
if not fixed_width_concentration_table.empty:
    concentration_envelope = fixed_width_concentration_table.groupby("Lx").agg(
        median_std=("basis_independent_std", "median"),
        max_std=("basis_independent_std", "max"),
        max_p90=("p90_abs_deviation", "max"),
        max_degenerate_fraction=("degenerate_state_fraction", "max"),
    ).reset_index()
    display(concentration_envelope)
    concentration_envelope.to_csv(DATA_DIR / "qdm_background_concentration_envelope.csv", index=False)
    fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
    ax.plot(concentration_envelope["Lx"], concentration_envelope["median_std"], marker="o", label="median local spread")
    ax.plot(concentration_envelope["Lx"], concentration_envelope["max_std"], marker="s", label="largest local spread")
    ax.set_xlabel(r"Strip length $L_x$")
    ax.set_ylabel("Window EEV spread")
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    save_figure(fig, "qdm_background_concentration")
    plt.show()

In [ ]:
# Exact beta-zero strip traces are kept as an analytical reference.  The
# energy-matched microcanonical ensemble in the next section is the actual ETH
# comparator for the lambda=1 cage energy.
transfer = SquareQDMStripTransferMatrix(circumference=4)
strip_rows = []
transfer_witnesses = [("A", a_placement), ("Z", z_placement)]
if Y_THERMAL_ENABLED:
    transfer_witnesses.append(("Y", y_placement))
for witness_name, placement in transfer_witnesses:
    scaling = transfer.scan_witness(
        placement,
        lengths=strip_lengths,
        boundary_x="periodic",
        winding_sector=(0, 0),
        winding_projection="fourier",
    )
    for evaluation in scaling.evaluations:
        strip_rows.append({
            "witness": witness_name,
            "length": evaluation.length,
            "circumference": evaluation.circumference,
            "thermal_activity": evaluation.expectation,
            "cage_activity": 0.0,
            "partition_count": evaluation.partition_count,
            "window_width": evaluation.window_width,
            "Y_validation_status": Y_VALIDATION_STATUS,
        })
three_witness_strip_table = pd.DataFrame(strip_rows)
three_witness_strip_table.to_csv(DATA_DIR / "qdm_three_witness_beta_zero_strip.csv", index=False)
display(three_witness_strip_table.head(12))

fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
for witness_name, marker in (("A", "o"), ("Z", "s"), ("Y", "^")):
    if witness_name == "Y" and not Y_THERMAL_ENABLED:
        continue
    subset = three_witness_strip_table[three_witness_strip_table["witness"] == witness_name]
    ax.plot(subset["length"], subset["thermal_activity"], marker=marker, label=rf"$Q_R^{witness_name}$")
ax.axhline(0.0, linestyle="--", linewidth=0.8, label="cage")
ax.set_xlabel(r"Strip length $L_x$")
ax.set_ylabel(r"$\mathrm{Tr}(\rho_{\beta=0}Q_R)$")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
save_figure(fig, "qdm_three_witness_beta_zero_activity")
plt.show()


The strip trace is a large-size computational extension only after the finite-size energy-density and local-observable overlap is checked. The revised $Y_R$ transfer sequence is generated only when the shell operator is dark in the resolved cage; otherwise the controlled $A_R,Z_R$ routes are retained and the $Y$ column remains explicitly unavailable.

### T1c. Fixed-width fit diagnostics

The exact-ED sequence is currently too short for a precision extrapolation.
The table nevertheless standardizes the fit forms and records an explicit
`insufficient_sizes` status until at least three controlled lengths are
available.


In [ ]:
fit_rows=[]
for witness,column in (("A","thermal_A_activity"),("Z","thermal_Z_activity"),("Y","thermal_Y_activity")):
    if witness=="Y" and not Y_THERMAL_ENABLED: continue
    frame=fixed_width_primary_table.sort_values("Lx"); xL=frame["Lx"].to_numpy(dtype=float); y=frame[column].to_numpy(dtype=float)
    for fit_form,x in (("a+b/Lx",1/xL),("a+b/Lx^2",1/xL**2)):
        if len(y)<3:
            fit_rows.append({"witness":witness,"fit_form":fit_form,"included_sizes":",".join(map(str,xL.astype(int))),
                             "limit":np.nan,"slope":np.nan,"rmse":np.nan,"status":"insufficient_sizes"})
        else:
            slope,intercept=np.polyfit(x,y,1); pred=intercept+slope*x
            fit_rows.append({"witness":witness,"fit_form":fit_form,"included_sizes":",".join(map(str,xL.astype(int))),
                             "limit":float(intercept),"slope":float(slope),"rmse":float(np.sqrt(np.mean((y-pred)**2))),
                             "status":"diagnostic_only_short_sequence"})
qdm_cage_excised_fit_summary=pd.DataFrame(fit_rows)
qdm_cage_excised_fit_summary.to_csv(DATA_DIR/"qdm_cage_excised_fit_summary.csv",index=False)
display(qdm_cage_excised_fit_summary)


## T2. Matched $\beta=0$ fixed-width extension

The transfer calculation is connected to the microcanonical result only after recording the finite-size cage--trace energy-density match and the microcanonical--normalized-trace differences for the same transported witnesses.

In [ ]:
if RUN_BETA0_OVERLAP:
    beta0_overlap_rows=[]
    for row in fixed_width_primary_table.itertuples(index=False):
        cache=qdm_primary_cache[int(row.repeats)]
        transfer_at_length=three_witness_strip_table[three_witness_strip_table["length"]==int(row.Lx)].set_index("witness")
        trace_energy_density=float(np.mean(cache["energies"])/row.volume)
        beta0_overlap_rows.append({
            "repeats":int(row.repeats),"Lx":int(row.Lx),"Ly":int(row.Ly),
            "cage_energy_density":float(row.cage_energy_density),"beta0_trace_energy_density":trace_energy_density,
            "energy_density_mismatch":abs(float(row.cage_energy_density)-trace_energy_density),
            "window_energy_density_half_width":float(row.window_energy_density_half_width),
            "window_state_count":int(row.window_state_count),"retained_state_count":int(row.retained_state_count),
            "removed_projector_rank":int(row.removed_projector_rank),"removed_fraction":float(row.removed_fraction),
            "tau_A_mc_th":float(row.thermal_A_activity),"tau_A_beta0":float(transfer_at_length.loc["A","thermal_activity"]),
            "delta_A":abs(float(row.thermal_A_activity)-float(transfer_at_length.loc["A","thermal_activity"])),
            "tau_Z_mc_th":float(row.thermal_Z_activity),"tau_Z_beta0":float(transfer_at_length.loc["Z","thermal_activity"]),
            "delta_Z":abs(float(row.thermal_Z_activity)-float(transfer_at_length.loc["Z","thermal_activity"])),
            "tau_Y_mc_th":float(row.thermal_Y_activity) if Y_THERMAL_ENABLED else np.nan,
            "tau_Y_beta0":float(transfer_at_length.loc["Y","thermal_activity"]) if Y_THERMAL_ENABLED else np.nan,
            "delta_Y":abs(float(row.thermal_Y_activity)-float(transfer_at_length.loc["Y","thermal_activity"])) if Y_THERMAL_ENABLED else np.nan,
            "Y_validation_status":Y_VALIDATION_STATUS,
        })
    qdm_beta0_overlap_df=pd.DataFrame(beta0_overlap_rows)
else:
    qdm_beta0_overlap_df=pd.DataFrame()
qdm_beta0_overlap_df.to_csv(DATA_DIR/"qdm_beta0_cage_excised_overlap.csv",index=False)
qdm_beta0_overlap_df.to_csv(DATA_DIR/"qdm_beta0_ensemble_overlap.csv",index=False)
display(qdm_beta0_overlap_df)


## C7. Exceptional-projector audit

Projector excision is already part of every T1 window.  This section exports a
compact compatibility table for older manuscript tooling.


In [ ]:
qdm_protocol_m_df=fixed_width_primary_table.copy()
qdm_protocol_m_df.to_csv(DATA_DIR/"qdm_protocol_M_vs_D.csv",index=False)
display(qdm_protocol_m_df)


## T3. Preserving deformation geometry and finite paths

In [ ]:
term_builder = SparseHamiltonianBuilder(
    backend="scipy",
    dtype=np.complex128,
    on_missing="raise",
)
kinetic_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator]).astype(np.complex128)
    for operator in square_build.kinetic_operators
)
potential_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator]).astype(np.complex128)
    for operator in square_build.potential_operators
)
if len(potential_term_matrices) != 2 * len(square_model.plaquette_ids()):
    raise RuntimeError("Expected two orientation projectors per square plaquette.")
plaquette_potential_matrices = tuple(
    potential_term_matrices[2 * index] + potential_term_matrices[2 * index + 1]
    for index in range(len(square_model.plaquette_ids()))
)
phase_tangent_operators = tuple(
    PlaquettePatternOperator.qdm_flip(
        layout=square_model.layout,
        lattice=square_model.lattice,
        plaquette_id=int(plaquette_id),
        coefficient=1.0j,
        reverse_coefficient=-1.0j,
    )
    for plaquette_id in square_model.plaquette_ids()
)
phase_tangent_matrices = tuple(
    term_builder.build(square_build.basis, [operator]).astype(np.complex128)
    for operator in phase_tangent_operators
)
all_local_term_matrices = kinetic_term_matrices + potential_term_matrices
plaquette_centers = tuple(
    tuple(float(value) for value in square_model.lattice.plaquette_anchor_cell(int(pid)))
    for pid in square_model.plaquette_ids()
)

compact_state = states_04[:, 0]
collective_state = states_04[:, 8]
deformation_alphabets = {
    "flip amplitudes": kinetic_term_matrices,
    "Peierls phases": phase_tangent_matrices,
    "flippability potentials": plaquette_potential_matrices,
    "amplitudes + phases": kinetic_term_matrices + phase_tangent_matrices,
}
state_targets = {
    "compact": compact_state,
    "collective": collective_state,
}

hierarchy_reports = {}
hierarchy_rows = []
singular_rows = []
conditioning_rows = []
for target_name, state in state_targets.items():
    support = np.flatnonzero(np.abs(state) > TOL)
    conditioning = cage_jacobian_conditioning_from_hamiltonian(
        square_build.hamiltonian,
        support,
        state,
        tolerance=TOL,
    )
    conditioning_rows.append({
        "target": target_name,
        **conditioning.to_summary_dict(),
    })
    for alphabet_name, perturbations in deformation_alphabets.items():
        report = cage_compatibility_hierarchy_from_hamiltonians(
            square_build.hamiltonian,
            perturbations,
            support,
            state,
            coefficient_field="real",
            tolerance=TOL,
        )
        hierarchy_reports[(target_name, alphabet_name)] = report
        hierarchy_rows.append({
            "target": target_name,
            "alphabet": alphabet_name,
            "obstruction_rank": report.first_order.rank,
            **report.to_summary_dict(),
        })
        singular_rows.extend(
            {
                "target": target_name,
                "alphabet": alphabet_name,
                "singular_index": singular_index,
                "singular_value": float(singular_value),
            }
            for singular_index, singular_value in enumerate(report.first_order.singular_values)
        )

hierarchy_table = pd.DataFrame(hierarchy_rows)
conditioning_table = pd.DataFrame(conditioning_rows)
singular_table = pd.DataFrame(singular_rows)
hierarchy_table.to_csv(DATA_DIR / "qdm_4x4_cage_obstruction_hierarchy.csv", index=False)
conditioning_table.to_csv(DATA_DIR / "qdm_4x4_cage_conditioning.csv", index=False)
singular_table.to_csv(DATA_DIR / "qdm_4x4_cage_obstruction_spectra.csv", index=False)
display(hierarchy_table)
display(conditioning_table[["target", "support_size", "cage_gap", "full_residual"]])

In [ ]:
fig, ax = plt.subplots(figsize=PRX_WIDE_FIGSIZE)
plot_table = hierarchy_table.copy()
x = np.arange(len(deformation_alphabets))
width = 0.36
for target_index, target_name in enumerate(("compact", "collective")):
    group = plot_table[plot_table["target"] == target_name].set_index("alphabet").loc[list(deformation_alphabets)]
    ax.bar(
        x + (target_index - 0.5) * width,
        group["first_order_compatible_dimension"],
        width=width,
        label=target_name,
    )
ax.set_xticks(x, list(deformation_alphabets), rotation=16, ha="right")
ax.set_ylabel("First-order compatible dimension")
ax.legend(frameon=False)
save_figure(fig, "qdm_4x4_deformation_compatible_dimensions")
plt.show()

fig, ax = plt.subplots(figsize=PRX_WIDE_FIGSIZE)
for (target_name, alphabet_name), group in singular_table.groupby(["target", "alphabet"], sort=False):
    if alphabet_name not in ("flip amplitudes", "Peierls phases"):
        continue
    ax.semilogy(
        group["singular_index"] + 1,
        np.maximum(group["singular_value"], 1.0e-16),
        marker="o",
        label=f"{target_name}: {alphabet_name}",
    )
ax.set_xlabel("Obstruction singular-value index")
ax.set_ylabel("Singular value")
ax.legend(frameon=False)
save_figure(fig, "qdm_4x4_deformation_obstruction_spectra")
plt.show()

### T3a. Controlled Peierls reference and gauge audit

In [ ]:
gauge_incidence = qdm_plaquette_link_gauge_matrix(square_model.lattice)
gauge_rank = int(np.linalg.matrix_rank(gauge_incidence, tol=RANK_TOL))
uniform_phase_pattern = np.ones(len(square_model.plaquette_ids()), dtype=np.float64)
gauge_projection = gauge_incidence @ np.linalg.lstsq(
    gauge_incidence,
    uniform_phase_pattern,
    rcond=RANK_TOL,
)[0]
uniform_non_gauge_residual = float(np.linalg.norm(uniform_phase_pattern - gauge_projection))

peierls_rows = []
for phase in np.linspace(-0.70, 0.70, 15):
    phase_model = replace(square_model, coup_kin=np.exp(1.0j * phase))
    phase_build = phase_model.build(
        basis_solver="dfs",
        builder="sparse",
        backend="scipy",
        sort_basis=True,
    )
    np.testing.assert_array_equal(phase_build.basis.states, square_build.basis.states)
    for target_name, state in state_targets.items():
        support = np.flatnonzero(np.abs(state) > TOL)
        eigenpair = diagnose_eigenpair(phase_build.hamiltonian, state)
        conditioning = cage_jacobian_conditioning_from_hamiltonian(
            phase_build.hamiltonian,
            support,
            state,
            tolerance=TOL,
        )
        peierls_rows.append({
            "phase": float(phase),
            "target": target_name,
            "energy": float(eigenpair.energy.real),
            "residual": eigenpair.residual_norm,
            "Delta_cage": conditioning.cage_gap,
        })

peierls_path_table = pd.DataFrame(peierls_rows)
peierls_path_table.to_csv(DATA_DIR / "qdm_4x4_uniform_peierls_path.csv", index=False)
pd.DataFrame([{
    "n_plaquette_phases": gauge_incidence.shape[0],
    "n_link_phases": gauge_incidence.shape[1],
    "link_gauge_phase_rank": gauge_rank,
    "uniform_phase_distance_from_link_gauge_subspace": uniform_non_gauge_residual,
}]).to_csv(DATA_DIR / "qdm_4x4_peierls_gauge_rank.csv", index=False)
display(peierls_path_table)

fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
for target_name, group in peierls_path_table.groupby("target"):
    ax.semilogy(
        group["phase"],
        np.maximum(group["residual"], 1.0e-16),
        marker="o",
        label=target_name,
    )
ax.axhline(TOL, linestyle="--", linewidth=0.8, label="tolerance")
ax.set_xlabel(r"Uniform plaquette phase $\phi$")
ax.set_ylabel("Fixed-vector residual")
ax.legend(frameon=False)
save_figure(fig, "qdm_uniform_peierls_compact_collective_residual")
plt.show()

fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
for target_name, group in peierls_path_table.groupby("target"):
    ax.plot(group["phase"], group["Delta_cage"], marker="o", label=target_name)
ax.set_xlabel(r"Uniform plaquette phase $\phi$")
ax.set_ylabel(r"Cage-conditioning gap $\Delta_{\rm cage}$")
ax.legend(frameon=False)
save_figure(fig, "qdm_uniform_peierls_cage_gap")
plt.show()

The tangent ranks and nonlinear path answer different questions. The uniform phase is used only as a reproducible thermal-reference deformation. Its distance from the link-gauge phase subspace is recorded numerically and determines whether it is physically non-gauge. Nontrivial robustness requires the separate non-gauge or nonuniform-potential paths below.

### T3b. Controlled Peierls thermal scan

In [ ]:
def peierls_thermal_point_4x4(phase):
    raw_instance = product_unit_cell.with_couplings(
        coup_kin=np.exp(1.0j * float(phase)),
        coup_pot=1.0,
    ).instantiate(1)
    phase_model = replace(raw_instance.model, winding_x=0, winding_y=0)
    instance = replace(raw_instance, model=phase_model)
    build = phase_model.build(
        basis_solver="dfs", builder="sparse", backend="scipy", sort_basis=True
    )
    configs = basis_configs_from_build_result(build)
    tx = square_qdm_basis_translation_permutation(phase_model, configs, dx=1)
    ty = square_qdm_basis_translation_permutation(phase_model, configs, dy=1)
    kx, ky = fixed_width_cage_momentum(1)
    sector = commuting_cyclic_symmetry_sector_basis(
        (tx, ty), orders=(4, 4), momentum_indices=(kx, ky)
    )
    cage = materialize_square_qdm_periodic_product_state(instance, configs)
    cage_sector = project_state_to_sector(cage, sector)
    cage_sector /= np.linalg.norm(cage_sector)
    h_sector = project_operator_to_sector(build.hamiltonian, sector)
    energies, vectors = scipy_linalg.eigh(h_sector, check_finite=False)

    witnesses = {
        "A": a_placement.instantiate_on_model(phase_model),
        "Z": z_placement.instantiate_on_model(phase_model),
    }
    if Y_THERMAL_ENABLED:
        witnesses["Y"] = y_placement.instantiate_on_model(phase_model)
    local = {name: witness.embed(configs) for name, witness in witnesses.items()}
    q_full = {name: operator.conj().T @ operator for name, operator in local.items()}
    q_sector = {name: project_operator_to_sector(operator, sector) for name, operator in q_full.items()}
    z_sector = project_operator_to_sector(local["Z"], sector)
    y_sector = project_operator_to_sector(local["Y"], sector) if Y_THERMAL_ENABLED else None

    cage_channel_means = {
        "Z": float(np.vdot(cage_sector, z_sector @ cage_sector).real),
        "Y": float(np.vdot(cage_sector, y_sector @ cage_sector).real) if y_sector is not None else np.nan,
    }
    controlled_means = [cage_channel_means["Z"]] + ([cage_channel_means["Y"]] if y_sector is not None else [])
    if max(abs(value) for value in controlled_means) > 1.0e-9:
        raise RuntimeError(f"Peierls path lost controlled Hermitian-channel darkness at phi={phase}: {cage_channel_means}")

    cage_report = diagnose_eigenpair(build.hamiltonian, cage)
    cage_energy = float(cage_report.energy.real)
    cage_activities = {
        name: float(np.vdot(cage_sector, operator @ cage_sector).real)
        for name, operator in q_sector.items()
    }
    if max(abs(value) for value in cage_activities.values()) > 1.0e-9:
        raise RuntimeError(f"Peierls path lost witness darkness at phi={phase}: {cage_activities}")

    plan = thermodynamic_energy_window_plan(
        volume=16,
        energy_density=cage_energy / 16.0,
        width_prefactor=PRIMARY_WINDOW_PREFACTOR,
        local_energy_scale=1.0,
    )
    window = select_microcanonical_window_by_width(
        energies,
        target_energy=cage_energy,
        half_width=plan.half_width,
        degeneracy_tolerance=TOL,
    )
    indices = np.asarray(window.indices, dtype=np.int64)
    qa = eigenstate_expectations(q_sector["A"], vectors)
    z_moments = spectral_observable_moments(z_sector, vectors, squared_operator=q_sector["Z"], indices=indices)
    y_moments = spectral_observable_moments(y_sector, vectors, squared_operator=q_sector["Y"], indices=indices) if y_sector is not None else None
    smooth_filter = gaussian_spectral_filter(
        energies, target_energy=cage_energy, sigma=SMOOTH_SIGMA_PREFACTOR * np.sqrt(16)
    )
    weights = np.asarray(smooth_filter.weights)
    conditioning = cage_jacobian_conditioning_from_hamiltonian(
        build.hamiltonian,
        np.flatnonzero(np.abs(cage) > TOL),
        cage,
        tolerance=TOL,
    )
    return {
        "phase": float(phase),
        "path_parameter": float(phase - PEIERLS_REFERENCE_PHASE),
        "cage_energy": cage_energy,
        "cage_residual": cage_report.residual_norm,
        "Delta_cage": conditioning.cage_gap,
        "window_state_count": window.n_states,
        "window_energy_density_half_width": plan.energy_density_half_width,
        "cage_QA": cage_activities["A"],
        "cage_QZ": cage_activities["Z"],
        "cage_QY": cage_activities.get("Y", np.nan),
        "cage_Z_mean": cage_channel_means["Z"],
        "cage_Y_mean": cage_channel_means["Y"],
        "thermal_A_activity": float(np.mean(qa[indices])),
        "thermal_Z_mean": z_moments.mean,
        "thermal_Z_activity": z_moments.second_moment,
        "thermal_Z_variance": z_moments.variance,
        "thermal_Y_mean": y_moments.mean if y_moments is not None else np.nan,
        "thermal_Y_activity": y_moments.second_moment if y_moments is not None else np.nan,
        "thermal_Y_variance": y_moments.variance if y_moments is not None else np.nan,
        "smooth_A_activity": float(np.dot(weights, eigenstate_expectations(q_sector["A"], vectors))),
        "smooth_Z_activity": float(np.dot(weights, eigenstate_expectations(q_sector["Z"], vectors))),
        "smooth_Y_activity": float(np.dot(weights, eigenstate_expectations(q_sector["Y"], vectors))) if Y_THERMAL_ENABLED else np.nan,
        "Y_validation_status": Y_VALIDATION_STATUS,
        "smooth_effective_state_count": smooth_filter.effective_state_count,
    }


peierls_thermal_table = pd.DataFrame([peierls_thermal_point_4x4(phase) for phase in PEIERLS_PATH])
peierls_thermal_table.to_csv(DATA_DIR / "qdm_4x4_peierls_three_witness_path.csv", index=False)
display(peierls_thermal_table)

fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
peierls_specs = [("thermal_A_activity", r"$Q_R^A$", "o"), ("thermal_Z_activity", r"$Q_R^Z$", "s")]
if Y_THERMAL_ENABLED:
    peierls_specs.append(("thermal_Y_activity", r"$Q_R^Y$", "^"))
for column, label, marker in peierls_specs:
    ax.plot(peierls_thermal_table["phase"], peierls_thermal_table[column], marker=marker, label=label)
ax.axvline(PEIERLS_REFERENCE_PHASE, linestyle="--", linewidth=0.8, color="0.4")
ax.set_xlabel(r"Uniform plaquette phase $\phi$")
ax.set_ylabel("Microcanonical activity")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
save_figure(fig, "qdm_peierls_three_witness_activity")
plt.show()

fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
ax.plot(peierls_thermal_table["phase"], peierls_thermal_table["thermal_Z_variance"], marker="s", label=r"${\rm Var}(Z_R)$")
ax.plot(peierls_thermal_table["phase"], np.abs(peierls_thermal_table["thermal_Z_mean"]), marker="o", label=r"$|\langle Z_R\rangle|$")
if Y_THERMAL_ENABLED:
    ax.plot(peierls_thermal_table["phase"], peierls_thermal_table["thermal_Y_variance"], marker="^", label=r"${\rm Var}(Y_R)$")
    ax.plot(peierls_thermal_table["phase"], np.abs(peierls_thermal_table["thermal_Y_mean"]), marker="v", label=r"$|\langle Y_R\rangle|$")
ax.set_xlabel(r"Uniform plaquette phase $\phi$")
ax.set_ylabel("Hermitian mean / variance")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
save_figure(fig, "qdm_peierls_hermitian_resolution")
plt.show()


### T3c. Nonuniform flippability-potential path

In [ ]:
def qdm_nonuniform_potential_point(g, *, repeats=1):
    raw = product_unit_cell.instantiate(int(repeats))
    model0 = replace(raw.model, winding_x=0, winding_y=0)
    # A deterministic, size-compatible but translation-breaking pattern.
    # Generic irrational frequencies avoid leaving an unrecorded momentum
    # symmetry in the deformed Hamiltonian.
    direction = {}
    for pid in model0.plaquette_ids():
        x, y = model0.lattice.plaquette_anchor_cell(int(pid))
        direction[int(pid)] = float(
            np.sin(np.sqrt(2.0) * (int(x) + 1) + np.sqrt(3.0) * (int(y) + 1))
            + 0.23 * np.cos(np.sqrt(5.0) * (int(x) + 1) - np.sqrt(7.0) * (int(y) + 1))
        )
    mean_direction = float(np.mean(list(direction.values())))
    direction = {pid: value - mean_direction for pid, value in direction.items()}
    potential = {pid: 1.0 + float(g) * direction[pid] for pid in direction}
    model = replace(model0, coup_kin=1.0, coup_pot=potential)
    instance = replace(raw, model=model)
    build = model.build(basis_solver="dfs", builder="sparse", backend="scipy", sort_basis=True)
    configs = basis_configs_from_build_result(build)
    cage = materialize_square_qdm_periodic_product_state(instance, configs)
    cage_report = diagnose_eigenpair(build.hamiltonian, cage)
    # The generic potential pattern breaks translation symmetry, so the exact
    # symmetry resolution consists only of the already imposed winding sector.
    cage_sector = cage / np.linalg.norm(cage)
    h_sector = build.hamiltonian.toarray()
    energies, vectors = scipy_linalg.eigh(h_sector, check_finite=False)
    # A and Z follow the kinetic boundary row.  The shifted potential
    # channel must instead follow the same nonuniform local coefficients:
    # Y_R(g)=v_1(g)F_{p_1}+v_2(g)F_{p_2}-[v_1(g)+v_2(g)]I.
    p1_anchor, p2_anchor = (0, 0), (0, 2)
    v1 = float(potential[next(int(pid) for pid in model.plaquette_ids() if tuple(model.lattice.plaquette_anchor_cell(int(pid))) == p1_anchor)])
    v2 = float(potential[next(int(pid) for pid in model.plaquette_ids() if tuple(model.lattice.plaquette_anchor_cell(int(pid))) == p2_anchor)])
    p1 = plaquette_local_positions(square_model, z_reference_witness.variable_indices, p1_anchor)
    p2 = plaquette_local_positions(square_model, z_reference_witness.variable_indices, p2_anchor)
    local_patterns = tuple(product((0, 1), repeat=z_reference_witness.template.n_variables))
    y_diagonal = np.asarray([
        v1 * flippability_on_pattern(pattern, p1)
        + v2 * flippability_on_pattern(pattern, p2)
        - (v1 + v2)
        for pattern in local_patterns
    ], dtype=np.complex128)
    y_continued = LocalWitnessTemplate(
        pattern_key=(),
        local_patterns=local_patterns,
        local_operator=np.diag(y_diagonal),
        metadata={
            "name": "Y_R_continued",
            "definition": "v1*F_p1+v2*F_p2-(v1+v2)I",
            "coefficients": (v1, v2),
        },
    ).normalized("operator_norm").instantiate(z_reference_witness.variable_indices)
    y_continued_placement = SquareQDMWitnessPlacement.from_local_witness(square_model, y_continued)
    witnesses = {
        "A": a_placement.instantiate_on_model(model),
        "Z": z_placement.instantiate_on_model(model),
        "Y": y_continued_placement.instantiate_on_model(model),
    }
    local = {name: witness.embed(configs) for name, witness in witnesses.items()}
    q_sector = {name: op.conj().T @ op for name, op in local.items()}
    cage_activities = {name: float(np.vdot(cage_sector, op @ cage_sector).real) for name, op in q_sector.items()}
    target = float(cage_report.energy.real)
    plan = thermodynamic_energy_window_plan(volume=model.lattice.num_plaquettes, energy_density=target/model.lattice.num_plaquettes, width_prefactor=PRIMARY_WINDOW_PREFACTOR, local_energy_scale=1.0)
    window = select_microcanonical_window_by_width(energies, target_energy=target, half_width=plan.half_width, degeneracy_tolerance=TOL)
    indices = np.asarray(window.indices, dtype=np.int64)
    activities = {name: float(np.mean(eigenstate_expectations(op, vectors)[indices])) for name, op in q_sector.items()}
    return {
        "g": float(g),
        "repeats": int(repeats),
        "Lx": int(model.lx),
        "cage_energy": target,
        "cage_residual": float(cage_report.residual_norm),
        "cage_QA": cage_activities["A"],
        "cage_QZ": cage_activities["Z"],
        "cage_QY": cage_activities["Y"],
        "resolved_symmetries": "winding_only_translation_broken",
        "window_state_count": int(window.n_states),
        "window_energy_density_half_width": float(plan.energy_density_half_width),
        "thermal_A_activity": activities["A"],
        "thermal_Z_activity": activities["Z"],
        "thermal_Y_activity": activities["Y"],
        "potential_pattern": repr(direction),
        "Y_p1_coefficient": v1,
        "Y_p2_coefficient": v2,
        "Y_definition": "v1*F_p1+v2*F_p2-(v1+v2)I",
    }

potential_path_rows = []
if RUN_NONUNIFORM_POTENTIAL_PATH:
    for g in np.unique(np.append(np.linspace(-0.30, 0.30, 7), NONUNIFORM_POTENTIAL_REFERENCE)):
        potential_path_rows.append(qdm_nonuniform_potential_point(g, repeats=1))
qdm_potential_path_df = pd.DataFrame(potential_path_rows)
qdm_potential_path_df.to_csv(DATA_DIR / "qdm_nonuniform_potential_path.csv", index=False)

# Stable T3 handoff filename.  This path currently uses the ordinary
# energy-matched window.  The complete continued exceptional projector and
# cage-excised concentration basis are still unfinished, so every exported row
# carries an explicit provisional protocol label.
qdm_deformed_grid_df = qdm_potential_path_df.copy()
if not qdm_deformed_grid_df.empty:
    qdm_deformed_grid_df["deformation_reference"] = "nonuniform-potential"
    qdm_deformed_grid_df["ensemble_protocol"] = "ordinary_microcanonical_window_provisional"
    qdm_deformed_grid_df["cage_excision_applied"] = False
    qdm_deformed_grid_df["exceptional_projector_status"] = "pending_complete_continued_projector"
    qdm_deformed_grid_df["background_concentration_status"] = "pending_local_basis_grid"
    qdm_deformed_grid_df["claim_status"] = "exact_darkness_and_positive_finite_size_activity_only"
else:
    qdm_deformed_grid_df = pd.DataFrame(columns=[
        "g", "deformation_reference", "ensemble_protocol",
        "cage_excision_applied", "exceptional_projector_status",
        "background_concentration_status", "claim_status",
    ])
qdm_deformed_grid_df.to_csv(DATA_DIR / "qdm_deformed_cage_excised_grid.csv", index=False)
display(qdm_potential_path_df)
if not qdm_potential_path_df.empty:
    fig = plt.figure(figsize=PRX_TWO_PANEL_FIGSIZE)
    grid = fig.add_gridspec(1, 2, wspace=0.30)
    ax0 = fig.add_subplot(grid[0, 0])
    ax1 = fig.add_subplot(grid[0, 1])
    ax0.semilogy(qdm_potential_path_df["g"], np.maximum(qdm_potential_path_df["cage_residual"], 1e-16), marker="o")
    ax0.axhline(TOL, linestyle="--", linewidth=0.8)
    ax0.set_xlabel("Potential deformation $g$")
    ax0.set_ylabel("Cage residual")
    for column, label, marker in (("thermal_A_activity", r"$Q^A$", "o"), ("thermal_Z_activity", r"$Q^Z$", "s"), ("thermal_Y_activity", r"$Q^Y$", "^")):
        ax1.plot(qdm_potential_path_df["g"], qdm_potential_path_df[column], marker=marker, label=label)
    ax1.set_xlabel("Potential deformation $g$")
    ax1.set_ylabel("Microcanonical activity")
    ax1.legend(frameon=False)
    for ax, panel in ((ax0, "(a)"), (ax1, "(b)")):
        ax.grid(alpha=0.25)
        add_panel_label(ax, panel)
    fig.tight_layout()
    save_figure(fig, "qdm_nonuniform_potential_path")
    plt.show()

### T3d. Non-gauge kinetic path

In [ ]:
non_gauge_rows = []
non_gauge_direction = None
if RUN_NON_GAUGE_KINETIC_PATH:
    # Reconstruct the exact repeatable 4x4 cage associated with the transported
    # A/Z/Y witness placement.  This need not coincide with records_04[:, 0].
    reference_raw = product_unit_cell.with_couplings(coup_kin=1.0, coup_pot=1.0).instantiate(1)
    reference_model = replace(reference_raw.model, winding_x=0, winding_y=0)
    reference_instance = replace(reference_raw, model=reference_model)
    reference_build = reference_model.build(basis_solver="dfs", builder="sparse", backend="scipy", sort_basis=True)
    reference_configs = basis_configs_from_build_result(reference_build)
    reference_cage = materialize_square_qdm_periodic_product_state(reference_instance, reference_configs)
    np.testing.assert_array_equal(reference_build.basis.states, square_build.basis.states)

    reference_phase_report = cage_compatibility_hierarchy_from_hamiltonians(
        reference_build.hamiltonian,
        phase_tangent_matrices,
        np.flatnonzero(np.abs(reference_cage) > TOL),
        reference_cage,
        coefficient_field="real",
        tolerance=TOL,
    ).first_order
    compatible = np.asarray(reference_phase_report.compatible_coefficient_basis, dtype=np.float64)
    gauge_u, gauge_s, _gauge_vh = np.linalg.svd(gauge_incidence, full_matrices=False)
    gauge_dimension = int(np.sum(gauge_s > RANK_TOL))
    gauge_basis = gauge_u[:, :gauge_dimension]
    gauge_projector = gauge_basis @ gauge_basis.T
    physical_components = (np.eye(gauge_incidence.shape[0]) - gauge_projector) @ compatible
    norms = np.linalg.norm(physical_components, axis=0)
    if norms.size and float(np.max(norms)) > 1.0e-8:
        non_gauge_direction = physical_components[:, int(np.argmax(norms))]
        non_gauge_direction /= np.linalg.norm(non_gauge_direction)
        plaquette_ids = tuple(int(pid) for pid in reference_model.plaquette_ids())
        for g in np.linspace(-0.35, 0.35, 7):
            couplings = {
                pid: np.exp(1.0j * float(g) * float(non_gauge_direction[index]))
                for index, pid in enumerate(plaquette_ids)
            }
            model = replace(reference_model, coup_kin=couplings, coup_pot=1.0)
            build = model.build(basis_solver="dfs", builder="sparse", backend="scipy", sort_basis=True)
            np.testing.assert_array_equal(build.basis.states, reference_build.basis.states)
            report = diagnose_eigenpair(build.hamiltonian, reference_cage)
            configs = basis_configs_from_build_result(build)
            local_witnesses = {
                "A": a_placement.instantiate_on_model(model),
                "Z": z_placement.instantiate_on_model(model),
                "Y": y_placement.instantiate_on_model(model),
            }
            local_ops = {name: witness.embed(configs) for name, witness in local_witnesses.items()}
            cage_activities = {
                name: float(np.vdot(reference_cage, (operator.conj().T @ operator) @ reference_cage).real)
                for name, operator in local_ops.items()
            }
            non_gauge_rows.append(
                {
                    "g": float(g),
                    "cage_residual": float(report.residual_norm),
                    "energy": float(report.energy.real),
                    "fixed_QA_residual": cage_activities["A"],
                    "fixed_QZ_residual": cage_activities["Z"],
                    "fixed_QY_residual": cage_activities["Y"],
                    "direction_distance_from_gauge_subspace": float(np.linalg.norm((np.eye(gauge_incidence.shape[0]) - gauge_projector) @ non_gauge_direction)),
                    "direction": repr(non_gauge_direction.tolist()),
                }
            )
qdm_non_gauge_path_df = pd.DataFrame(
    non_gauge_rows,
    columns=[
        "g", "cage_residual", "energy",
        "fixed_QA_residual", "fixed_QZ_residual", "fixed_QY_residual",
        "direction_distance_from_gauge_subspace", "direction",
    ],
)
qdm_non_gauge_path_df.to_csv(DATA_DIR / "qdm_non_gauge_kinetic_path.csv", index=False)
display(qdm_non_gauge_path_df if not qdm_non_gauge_path_df.empty else pd.DataFrame({"status": ["no non-gauge compatible tangent found"]}))
if not qdm_non_gauge_path_df.empty:
    fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
    ax.semilogy(qdm_non_gauge_path_df["g"], np.maximum(qdm_non_gauge_path_df["cage_residual"], 1e-16), marker="o", label="cage")
    ax.semilogy(qdm_non_gauge_path_df["g"], np.maximum(qdm_non_gauge_path_df[["fixed_QA_residual", "fixed_QZ_residual", "fixed_QY_residual"]].max(axis=1), 1e-16), marker="s", label="largest fixed-channel residual")
    ax.axhline(TOL, linestyle="--", linewidth=0.8)
    ax.set_xlabel("Non-gauge kinetic path parameter $g$")
    ax.set_ylabel("Residual")
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    save_figure(fig, "qdm_non_gauge_kinetic_path")
    plt.show()

## Secondary: compact-versus-collective cancellation range

In [ ]:
term_builder = SparseHamiltonianBuilder(
    backend="scipy",
    dtype=np.complex128,
    on_missing="raise",
)
kinetic_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator])
    for operator in square_build.kinetic_operators
)
potential_term_matrices = tuple(
    term_builder.build(square_build.basis, [operator])
    for operator in square_build.potential_operators
)
all_local_term_matrices = kinetic_term_matrices + potential_term_matrices
plaquette_centers = tuple(
    tuple(float(value) for value in square_model.lattice.plaquette_anchor_cell(int(pid)))
    for pid in square_model.plaquette_ids()
)

radius_scans = {}
for label, state in (
    ("compact record 0", states_04[:, 0]),
    ("collective record 8", states_04[:, 8]),
):
    radius_scans[label] = scan_windowed_operator_annihilators(
        kinetic_term_matrices,
        state,
        plaquette_centers,
        radii=(0, 1, 2),
        periodic_box=(4, 4),
        metric="chebyshev",
        normalize_actions=True,
        action_tolerance=1.0e-12,
        rank_tolerance=TOL,
    )

radius_rows = [
    {"state": label, **point.to_summary_dict()}
    for label, report in radius_scans.items()
    for point in report.points
]
radius_table = pd.DataFrame(radius_rows)
radius_table.to_csv(DATA_DIR / "qdm_4x4_minimum_annihilator_radius.csv", index=False)
display(radius_table[[
    "state", "radius", "minimum_residual", "n_active", "rank", "nullity",
    "coefficient_support_size", "active_operator_indices",
]])

In [ ]:
fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
for label, report in radius_scans.items():
    ax.semilogy(
        [point.radius for point in report.points],
        [max(point.minimum_residual, 1.0e-16) for point in report.points],
        marker="o",
        label=label,
    )
ax.axhline(TOL, linestyle="--", linewidth=0.8, label="numerical tolerance")
ax.set_xlabel("Allowed Chebyshev radius")
ax.set_ylabel("Minimum annihilation residual")
ax.set_xticks([0, 1, 2])
ax.legend(frameon=False)
save_figure(fig, "qdm_4x4_annihilator_radius")
plt.show()

The compact representative first reaches the numerical kernel at radius one using two active plaquette terms.  The collective quotient retains a residual about $0.675$ at radius one and reaches a kernel only when all 16 plaquette terms are available.  On the $4\times4$ torus, this is evidence for a system-scale cancellation, not a proof that its radius must diverge on every possible continuation.

## Secondary: compact-versus-collective deformation rank

In [ ]:
state_resolved_reports = {}
state_resolved_rows = []
state_resolved_singular_rows = []
state_operator_basis = kinetic_term_matrices + phase_tangent_matrices
for label, state in (("compact", compact_state), ("collective", collective_state)):
    report = operator_coefficient_compatibility(
        state_operator_basis,
        state,
        mode="fixed_vectors",
        tolerance=RANK_TOL,
    )
    state_resolved_reports[label] = report
    state_resolved_rows.append(
        {
            "target": label,
            "n_operators": report.n_operators,
            "compatible_dimension": report.compatible_dimension,
            "obstruction_rank": report.rank,
            "singular_gap": report.singular_gap,
        }
    )
    state_resolved_singular_rows.extend(
        {
            "target": label,
            "singular_index": int(i),
            "singular_value": float(value),
        }
        for i, value in enumerate(report.singular_values)
    )

state_resolved_table = pd.DataFrame(state_resolved_rows)
state_resolved_spectrum_table = pd.DataFrame(state_resolved_singular_rows)
state_resolved_table.to_csv(DATA_DIR / "qdm_state_resolved_preserving_dimensions.csv", index=False)
state_resolved_spectrum_table.to_csv(DATA_DIR / "qdm_state_resolved_singular_spectra.csv", index=False)
display(state_resolved_table)

# Fixed-state joint continuation is immediate: if the same state is retained,
# the already-dark fixed local operators remain dark.  The harder rotating-
# state joint Jacobian remains explicitly unresolved.
qdm_joint_fixed_state_table = hierarchy_table[[
    "target", "alphabet", "n_parameters",
    "first_order_compatible_dimension",
    "fixed_state_compatible_dimension",
]].copy()
qdm_joint_fixed_state_table["joint_fixed_state_channel_dimension"] = qdm_joint_fixed_state_table["fixed_state_compatible_dimension"]
qdm_joint_fixed_state_table["rotating_state_joint_jacobian_status"] = "not_yet_computed"
qdm_joint_fixed_state_table.to_csv(DATA_DIR / "qdm_joint_fixed_state_channel_dimensions.csv", index=False)
display(qdm_joint_fixed_state_table)

fig, ax = plt.subplots(figsize=PRX_SINGLE_PANEL_FIGSIZE)
for label, marker in (("compact", "o"), ("collective", "s")):
    subset = state_resolved_spectrum_table[state_resolved_spectrum_table["target"] == label]
    ax.semilogy(subset["singular_index"], subset["singular_value"], marker=marker, label=label)
ax.set_xlabel("Singular-value index")
ax.set_ylabel("State-resolved obstruction spectrum")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
save_figure(fig, "qdm_state_resolved_singular_spectra")
plt.show()


## Secondary: collective multi-row locality scan

In [ ]:
collective_cluster_rows = []
if RUN_COLLECTIVE_CLUSTER_SCAN:
    report = radius_scans["collective record 8"]
    for point in report.points:
        collective_cluster_rows.append(
            {
                "Lx": 4,
                "Ly": 4,
                "radius": int(point.radius),
                "n_active_plaquettes": int(point.n_active),
                "nullity": int(point.nullity),
                "minimum_residual": float(point.minimum_residual),
                "coefficient_support_size": int(point.coefficient_support_size),
                "uses_all_plaquettes": bool(point.coefficient_support_size == len(square_model.plaquette_ids())),
                "bounded_local_certificate": bool(point.minimum_residual <= TOL and point.coefficient_support_size < len(square_model.plaquette_ids())),
            }
        )
collective_cluster_df = pd.DataFrame(collective_cluster_rows)
collective_cluster_df.to_csv(DATA_DIR / "qdm_collective_cluster_locality_status.csv", index=False)
display(collective_cluster_df)

## Draft-ready fixed-width and deformation figure

The manuscript-scale renderer uses the controlled $A_R,Z_R$ fixed-width data and adds $Y_R$ only when the revised symmetry-resolved darkness test passes. Translation-breaking potential paths may still carry a valid continued shifted-potential channel even when the translation-projected reference cage does not.

In [ ]:
fig = plt.figure(figsize=PRX_TWO_PANEL_FIGSIZE)
grid = fig.add_gridspec(1, 2, wspace=0.30)
ax0 = fig.add_subplot(grid[0, 0])
ax1 = fig.add_subplot(grid[0, 1])

combined_specs = [("thermal_A_activity", r"$Q_R^A$", "o"), ("thermal_Z_activity", r"$Q_R^Z$", "s")]
if Y_THERMAL_ENABLED:
    combined_specs.append(("thermal_Y_activity", r"$Q_R^Y$", "^"))
for column, label, marker in combined_specs:
    ax0.plot(fixed_width_primary_table["Lx"], fixed_width_primary_table[column], marker=marker, label=label)
ax0.axhline(0.0, linestyle="--", linewidth=0.8, label="cage")
ax0.set_xlabel(r"Strip length $L_x$")
ax0.set_ylabel("Microcanonical activity")
use_integer_ticks(ax0, axis="x")
ax0.set_xticks(fixed_width_primary_table["Lx"].to_numpy(dtype=int))
ax0.grid(alpha=0.3)
ax0.legend(loc="upper right", frameon=False)
add_panel_label(ax0, "(a)")

for column, label, marker in combined_specs:
    ax1.plot(peierls_thermal_table["phase"], peierls_thermal_table[column], marker=marker, label=label)
ax1.axvline(PEIERLS_REFERENCE_PHASE, linestyle="--", linewidth=0.8, color="0.4")
ax1.set_xlabel(r"Uniform plaquette phase $\phi$")
ax1.set_ylabel("Microcanonical activity")
ax1.grid(alpha=0.3)
ax1.legend(loc="upper right", frameon=False)
add_panel_label(ax1, "(b)")

fig.tight_layout()
save_figure(fig, "qdm_fixed_width_and_peierls_three_witness")
plt.show()


## Evidence status represented in this notebook

**Primary:** undeformed cage-excised $A/Z$ ETH scatter and fixed-width sequence,
window/projector systematics, local-background concentration, and finite-size
matching to the resolved $\beta=0$ trace before transfer continuation.

**Provisional:** every shifted-shell $Y_R$ result remains conditional on the
resolved-cage validation.  The equal-flippability difference is never silently
substituted for the Section-3 shell operator.

**Secondary:** Peierls cleanup, non-gauge kinetic and nonuniform-potential
paths, compact/collective obstruction ranks, and the collective locality scan.
Dense ED beyond $8\times4$ is not enabled by the production profile.


## Output manifest

In [ ]:
manifest = sorted(
    str(path.relative_to(DATA_DIR))
    for path in DATA_DIR.rglob("*")
    if path.is_file()
)
for item in manifest:
    print(item)
figure_manifest = write_figure_manifest(DATA_DIR / "figure_manifest.json")
display(figure_manifest)

claim_manifest = pd.DataFrame([
    {"claim_id":"C1-C2 A/Z","stage":"exact channels","status":"established_fixed_width","primary_file":"qdm_three_witness_certificates.csv"},
    {"claim_id":"C1-C2 Y","stage":"revised shifted shell","status":Y_VALIDATION_STATUS,"primary_file":"qdm_revised_Y_size_validation.csv"},
    {"claim_id":"C3a-C7","stage":"T1 undeformed cage-excised","status":"short_fixed_width_sequence","primary_file":"qdm_cage_excised_fixed_width.csv"},
    {"claim_id":"C3b","stage":"T2 beta0 matching","status":"finite_size_overlap_plus_transfer","primary_file":"qdm_beta0_cage_excised_overlap.csv"},
    {"claim_id":"C4-C6","stage":"T3 preserving deformation","status":"finite_path_ordinary_window_cage_excision_pending","primary_file":"qdm_deformed_cage_excised_grid.csv"},
])
claim_manifest.to_csv(DATA_DIR/"claim_manifest.csv",index=False)
display(claim_manifest)
